### 牛顿法 Newton's method 

#### 二阶模型
在 $x_k$ 处作二阶泰勒展开

$$ f(x_k+p)\approx f(x_k)+\nabla f(x_k)^T p+ \frac{1}{2} p^T \nabla ^2 f(x_k) p  \tag{1}$$

注：$p$ 是向量(在不影响理解下，这里没有加粗变量), $((x_k+p)-x_k)^2$ 应表述为向量内积 $p^T p$

记 $g_k=\nabla f(x_k)$ , $H_k=\nabla ^2 f(x_k)$

对(1)式 关于 p 求导：
$$ g_k+H_k p=0 \tag{2}$$

若$H_k$可逆，牛顿方向满足：
$$ H_kp_k=-g_k \tag{3}$$

#### 下降方向条件

若 $H_k \succ 0$, 则：

$$ g_k^T p_k=-g_k^T H_k ^{-1} g_k <0 \tag{4}$$


对式(3)左乘 $p^T$,

$$f(x_k+p)-f(x_k) \approx g_k ^T p_k -\frac{1}{2}p^T g_k \tag{5}$$

对式(5)右端，可以按照(4)式判断，即牛顿方向 （在$x_k$处搜索到的$p_k$） 是下降方向 


存在两个问题：（1）某个位置的 $H_k$ 可能不正定； （2）因为使用二阶近似，远离最优时，二阶局部模型可能不准确。


问题（1）$H_k$ 不正定，牛顿方向无法保证下降（可能指向鞍点或上升方向），常见处理方法：

（1）使用 $H_k+\delta I$进行正则化；
（2）采用修正 Cholesky分解；
（3）若方向不能保证下降，就使用负梯度方向（即梯度下降）；
（4）改用信赖域方法。

补充：正定二次型等函数，是处处$ H =A \succ 0$, 可以直接使用普通牛顿法。

#### 阻尼牛顿法

纯 牛顿法 使用 $\alpha_k=1$, 更新 $x_{k+1}=x_k+p_k$

因为问题（2）存在， 采用阻尼牛顿法 （带线搜索的牛顿法）：

$$ x_{k+1}=x_k+\alpha_k p_k \tag{6}$$

先从二阶近似模型得到牛顿方向，$H_k p_k=-g_k$

再由真实函数判断其是否是下降方向：

将$f(x_k+\alpha_k p_k)$ 看作关于 $\alpha$的 函数 $\phi(\alpha)$, 有

$$ \begin{aligned} \phi '(0) &= lim_{\alpha \to 0^+} \frac{f(x_k+\alpha p)-f(x_k)}{\alpha}\\
&=g_k^T p<0 \end{aligned} \tag{7}$$


这也是 f 在 p 方向的方向导数，$<0$表示沿着 p 走出极小一步 (即$\alpha>0$ 足够小), 能够保证函数下降


余下问题是，如何判断，当前点距离最优点远近，用二阶近似模型是否准确？
这里暂不用信赖域的方法，直接使用 `gd_Armijo` 提到Armijo条件，只要：

$$ f(x_k+\alpha p_k)\le f(x_k)+c_1 \alpha g_k^T p_k \tag{8}$$



In [6]:
#import importlib
import model
#importlib.reload(model)

In [7]:
import numpy as np
from time import perf_counter

In [8]:
#阻尼牛顿法 

def damped_newton(features,labels,max_iter=30,tolerance=1e-9):
    weights=np.zeros(features.shape[1])
    start_time=perf_counter()
    for iter in range(1, max_iter+1):
        grad=model.gradient(weights,features,labels)
        if np.linalg.norm(grad)<=tolerance:
            break
        direction=np.linalg.solve(model.hessian(weights,features,labels),-grad) #解方程 H_k p=-g_k
        alpha,local_steps=armijo(weights,direction,features,labels)
        weights+=alpha*direction
    return model.make_record("Damped-Newton",weights,features,labels,iter,perf_counter()-start_time)

In [9]:
def armijo(weights,direction,features,labels,initial_alpha=1.0,c1=1e-4,shrink=0.5):
    # 条件不成立  \alpha 收缩
    value=model.objective(weights,features,labels)
    grad=model.gradient(weights,features,labels)

    # \nabla f(x_k)^T p_k<0  p_k下降方向
    slope=np.dot(grad,direction)
    if slope>=0.0:
        raise ValueError("Armijo line search requires a descent direction")
    alpha=initial_alpha
    iters=0

    # f(x_{k+1})<=f(x_k)+c_1 \alpha_k \nabla f(x_k)^Tp_k
    while model.objective(weights+alpha*direction,features,labels)>value+c1*alpha*slope:
        alpha*=shrink
        iters+=1
        if alpha<1e-14:
            raise RuntimeError("Armijo line search failed: alpha is too small")
    return alpha, iters+1

In [10]:
def test_damped_newton():
    x_train, y_train, x_test, y_test, metadata = model.load_data()
    results=damped_newton(x_train,y_train)
    accuracy, precision,recall, log_loss = model.classification_metrics(
            results["weights"], x_test, y_test)
    print(
        f"method: {results.get('method')}\n"
        f"accuracy: {accuracy:.4f}\n"
        f"precision: {precision:.4f}\n"
        f"recall: {recall:.4f}\n"
        f"log_loss: {log_loss:.4f}"
    )

test_damped_newton()


method: Damped-Newton
accuracy: 0.9781
precision: 0.9586
recall: 0.8968
log_loss: 0.0619


小结：一般而言，牛顿法比梯度下降要快，主要是H对方向的修正，以及靠近最优点时收敛阶更高 (梯度下降通常是线性收敛，牛顿法能达到二次收敛）。

示例： $f(x,y)=100x^2+y^2$ 

梯度 $\nabla f=(200x,2y)$ 这表明在x方向很陡，如果步长稍大容易来回震荡，所以必须取很小的学习率，这导致在y方向又走得很慢。

$$ H=\begin{bmatrix} 200 &0\\
        0 &2 \end{bmatrix}$$

牛顿方向：$p=-H^{-1}g=(-x, -y)^T$

$(x,y)+p=(0,0)$ 对于严格凸二次函数，牛顿法能一步到达最优点。
